# CuTeDSL 02: CuTe DSL 布局代数

深入理解 CuTe 的布局系统：
- **Coalesce**：布局合并操作
- **Composition**：布局组合操作
- **Division**：布局划分操作（logical_divide, zipped_divide, tiled_divide, flat_divide）
- **Product**：布局乘积操作（logical_product, blocked_product, raked_product）
- **组合布局**：使用 inner/offset/outer 实现复杂数据变换
- **Gather/Scatter**：使用组合布局实现间接访问模式


## 一、布局代数操作


参考 CuTe C++ 的 [01_layout.md](https://github.com/NVIDIA/cutlass/blob/main/media/docs/cpp/cute/01_layout.md) 和 [02_layout_algebra.md](https://github.com/NVIDIA/cutlass/blob/main/media/docs/cpp/cute/02_layout_algebra.md) 文档。

在 CuTe 中，`Layout` 由 `Shape` 和 `Stride` 定义，将坐标映射到索引，支持静态/动态值。布局代数包括：Composition（组合）、Divide（划分）、Product（乘积）。本 notebook 演示 Python DSL 中的核心操作及静态/动态布局的打印差异。详见 [02_layout_algebra.md](https://github.com/NVIDIA/cutlass/blob/main/media/docs/cpp/cute/02_layout_algebra.md)。

In [2]:
import cutlass
import cutlass.cute as cute

布局代数支持：高效数据分块、线程/数据布局分离、tensor core 程序的分层描述、混合静态/动态转换、与 tensor 操作无缝集成、将 MMA/拷贝表达为规范循环。

### 1. Coalesce（合并）

`coalesce` 操作通过在可能时展平和合并模式来简化布局，同时不改变其大小或作为整数函数的行为。

它确保以下后置条件：
- 保持大小：cute.size(layout) == cute.size(result)，
- 展平：depth(result) <= 1，
- 保持函数性：对于所有 i，0 <= i < cute.size(layout)，layout(i) == result(i)。

基本 Coalesce 示例：

In [3]:
@cute.jit
def coalesce_example():
    """
    演示 coalesce 操作展平和合并模式
    """
    layout = cute.make_layout(
        (2, (1, 6)), stride=(1, (cutlass.Int32(6), 2))
    )  # 动态 stride
    result = cute.coalesce(layout)

    print(">>> Original:", layout)
    cute.printf(">?? Original: {}", layout)
    print(">>> Coalesced:", result)
    cute.printf(">?? Coalesced: {}", result)


coalesce_example()

>>> Original: (2,(1,6)):(1,(?,2))
>>> Coalesced: 12:1


>?? Original: (2,(1,6)):(1,(6,2))
>?? Coalesced: 12:1


In [4]:
@cute.jit
def coalesce_post_conditions():
    """
    演示 coalesce 操作的 3 个后置条件：
    1. size(@a result) == size(@a layout)
    2. depth(@a result) <= 1
    3. 对于所有 i，0 <= i < size(@a layout)，@a result(i) == @a layout(i)
    """
    layout = cute.make_layout(
        ((2, (3, 4)), (3, 2), 1), stride=((4, (8, 24)), (2, 6), 12)
    )
    result = cute.coalesce(layout)

    print(">>> Original:", layout)
    print(">>> Coalesced:", result)

    print(">>> Checking post-conditions:")
    print(">>> 1. Checking size remains the same after the coalesce operation:")
    original_size = cute.size(layout)
    coalesced_size = cute.size(result)
    print(f"Original size: {original_size}, Coalesced size: {coalesced_size}")
    assert coalesced_size == original_size, (
        f"Size mismatch: original {original_size}, coalesced {coalesced_size}"
    )

    print(">>> 2. Checking depth of coalesced layout <= 1:")
    depth = cute.depth(result)
    print(f"Depth of coalesced layout: {depth}")
    assert depth <= 1, f"Depth of coalesced layout should be <= 1, got {depth}"

    print(
        ">>> 3. Checking layout functionality remains the same after the coalesce operation:"
    )
    for i in cutlass.range_constexpr(original_size):
        original_value = layout(i)
        coalesced_value = result(i)
        print(f"Index {i}: original {original_value}, coalesced {coalesced_value}")
        assert coalesced_value == original_value, (
            f"Value mismatch at index {i}: original {original_value}, coalesced {coalesced_value}"
        )

coalesce_post_conditions()

>>> Original: ((2,(3,4)),(3,2),1):((4,(8,24)),(2,6),12)
>>> Coalesced: (24,6):(4,2)
>>> Checking post-conditions:
>>> 1. Checking size remains the same after the coalesce operation:
Original size: 144, Coalesced size: 144
>>> 2. Checking depth of coalesced layout <= 1:
Depth of coalesced layout: 1
>>> 3. Checking layout functionality remains the same after the coalesce operation:
Index 0: original 0, coalesced 0
Index 1: original 4, coalesced 4
Index 2: original 8, coalesced 8
Index 3: original 12, coalesced 12
Index 4: original 16, coalesced 16
Index 5: original 20, coalesced 20
Index 6: original 24, coalesced 24
Index 7: original 28, coalesced 28
Index 8: original 32, coalesced 32
Index 9: original 36, coalesced 36
Index 10: original 40, coalesced 40
Index 11: original 44, coalesced 44
Index 12: original 48, coalesced 48
Index 13: original 52, coalesced 52
Index 14: original 56, coalesced 56
Index 15: original 60, coalesced 60
Index 16: original 64, coalesced 64
Index 17: original 68

按模式 Coalesce 示例：

In [5]:
@cute.jit
def bymode_coalesce_example():
    """
    演示按模式合并
    """
    layout = cute.make_layout((2, (1, 6)), stride=(1, (6, 2)))

    # 使用模式配置 (1,1) 进行合并 = 合并两个模式
    result = cute.coalesce(layout, target_profile=(1, 1))

    # 打印结果
    print(">>> Original: ", layout)
    print(">>> Coalesced Result: ", result)


bymode_coalesce_example()

>>> Original:  (2,(1,6)):(1,(6,2))
>>> Coalesced Result:  (2,6):(1,2)


### 2. Composition（组合）

布局 `A` 与布局 `B` 的 `Composition`（组合）创建一个新布局 `R = A ◦ B`，其中：
- `B` 的形状与 `R` 的形状兼容，使得 `B` 的所有坐标也可以用作 `R` 的坐标，
- 对于 `B` 域中的所有坐标 `c`，`R(c) = A(B(c))`。

布局组合对于重塑和重新排序布局非常有用。

基本 Composition 示例：

In [6]:
@cute.jit
def composition_example():
    """
    演示基本布局组合 R = A ◦ B
    """
    A = cute.make_layout((6, 2), stride=(cutlass.Int32(8), 2))  # 动态 stride
    B = cute.make_layout((4, 3), stride=(3, 1))
    R = cute.composition(A, B)

    # 打印静态和动态信息
    print(">>> Layout A:", A)
    cute.printf(">?? Layout A: {}", A)
    print(">>> Layout B:", B)
    cute.printf(">?? Layout B: {}", B)
    print(">>> Composition R = A ◦ B:", R)
    cute.printf(">?? Composition R: {}", R)


composition_example()

>>> Layout A: (6,2):(?,2)
>>> Layout B: (4,3):(3,1)
>>> Composition R = A ◦ B: ((2,2),3):((?{div=3},2),?)
>?? Layout A: (6,2):(8,2)
>?? Layout B: (4,3):(3,1)
>?? Composition R: ((2,2),3):((24,2),8)


静态和动态布局的 Composition 比较：

在这种情况下，结果可能看起来不同，但在数学上是相同的。shape 中的 1 不会影响布局作为整数上的数学函数。在动态情况下，CuTe 无法合并动态的 size-1 模式来"简化"布局，因为对于该参数在运行时可能实现的所有可能动态值，这样做并不有效。

In [7]:
@cute.jit
def composition_static_vs_dynamic_layout():
    """
    展示静态和动态组合结果的差异
    """
    # 静态版本 - 使用编译时值
    A_static = cute.make_layout((10, 2), stride=(16, 4))
    B_static = cute.make_layout((5, 4), stride=(1, 5))
    R_static = cute.composition(A_static, B_static)

    # 静态打印显示编译时信息
    print(">>> Static composition:")
    print(">>> A_static: ", A_static)
    print(">>> B_static: ", B_static)
    print(">>> R_static: ", R_static)

    # 动态版本 - 使用运行时 Int32 值
    A_dynamic = cute.make_layout(
        (cutlass.Int32(10), cutlass.Int32(2)),
        stride=(cutlass.Int32(16), cutlass.Int32(4)),
    )
    B_dynamic = cute.make_layout(
        (cutlass.Int32(5), cutlass.Int32(4)),
        stride=(cutlass.Int32(1), cutlass.Int32(5)),
    )
    R_dynamic = cute.composition(A_dynamic, B_dynamic)

    # 动态 printf 显示运行时值
    cute.printf(">?? Dynamic composition:")
    cute.printf(">?? A_dynamic: {}", A_dynamic)
    cute.printf(">?? B_dynamic: {}", B_dynamic)
    cute.printf(">?? R_dynamic: {}", R_dynamic)


composition_static_vs_dynamic_layout()

>>> Static composition:
>>> A_static:  (10,2):(16,4)
>>> B_static:  (5,4):(1,5)
>>> R_static:  (5,(2,2)):(16,(80,4))
>?? Dynamic composition:
>?? A_dynamic: (10,2):(16,4)
>?? B_dynamic: (5,4):(1,5)
>?? R_dynamic: ((5,1),(2,2)):((16,4),(80,4))


按模式 Composition 示例：

按模式组合允许我们对布局的各个模式单独应用组合操作。当你想独立操作特定模式布局（例如行和列）时，这特别有用。

在 CuTe 的上下文中，按模式组合通过使用 `Tiler` 实现，它可以是一个布局或布局元组。`Tiler` 元组的叶子指定如何组合目标布局的对应模式，允许独立处理子布局。

In [8]:
@cute.jit
def bymode_composition_example():
    """
    演示使用 tiler 的按模式组合
    """
    # Define the original layout A
    A = cute.make_layout(
        (cutlass.Int32(12), (cutlass.Int32(4), cutlass.Int32(8))),
        stride=(cutlass.Int32(59), (cutlass.Int32(13), cutlass.Int32(1))),
    )

    # 定义用于按模式组合的 tiler
    tiler = (3, 8)  # Apply 3:1 to mode-0 and 8:1 to mode-1

    # 应用按模式组合
    result = cute.composition(A, tiler)

    # 打印静态和动态信息
    print(">>> Layout A:", A)
    cute.printf(">?? Layout A: {}", A)
    print(">>> Tiler:", tiler)
    cute.printf(">?? Tiler: {}", tiler)
    print(">>> By-mode Composition Result:", result)
    cute.printf(">?? By-mode Composition Result: {}", result)


bymode_composition_example()

>>> Layout A: (?,(?,?)):(?,(?,?))
>>> Tiler: (3, 8)
>>> By-mode Composition Result: (3,(?,?)):(?,(?,?))
>?? Layout A: (12,(4,8)):(59,(13,1))
>?? Tiler: (3,8)
>?? By-mode Composition Result: (3,(4,2)):(59,(13,1))


### 3. Division（划分为分块）

CuTe 中的 Division 操作用于将布局拆分为分块，这对于在线程或内存层次之间分区数据特别有用。

Logical divide（逻辑划分）：

当应用于两个布局时，`logical_divide` 将布局拆分为两个模式——第一个模式包含 tiler 指向的元素，第二个模式包含剩余元素。

In [9]:
@cute.jit
def logical_divide_1d_example():
    """
    演示一维逻辑划分
    """
    # Define the original layout
    layout = cute.make_layout((4, 2, 3), stride=(2, 1, 8))  # (4,2,3):(2,1,8)

    # Define the tiler
    tiler = cute.make_layout(4, stride=2)  # Apply to layout 4:2

    # Apply logical divide
    result = cute.logical_divide(layout, tiler=tiler)

    # Print results
    print(">>> Layout:", layout)
    print(">>> Tiler :", tiler)
    print(">>> Logical Divide Result:", result)
    cute.printf(">?? Logical Divide Result: {}", result)


logical_divide_1d_example()

>>> Layout: (4,2,3):(2,1,8)
>>> Tiler : 4:2
>>> Logical Divide Result: ((2,2),(2,3)):((4,1),(2,8))
>?? Logical Divide Result: ((2,2),(2,3)):((4,1),(2,8))


当应用于一个 Layout 和一个 `Tiler` 元组时，`logical_divide` 将自身应用于 `Tiler` 的叶子节点和目标 Layout 的对应模式。这意味着子布局根据 `Tiler` 中的布局独立进行拆分。

In [10]:
@cute.jit
def logical_divide_2d_example():
    """
    演示二维逻辑划分：
    布局形状：(M, N, L, ...)
    Tiler 形状：<TileM, TileN>
    结果形状：((TileM,RestM), (TileN,RestN), L, ...)
    """
    # Define the original layout
    layout = cute.make_layout(
        (9, (4, 8)), stride=(59, (13, 1))
    )  # (9,(4,8)):(59,(13,1))

    # Define the tiler
    tiler = (
        cute.make_layout(3, stride=3),  # Apply to mode-0 layout 3:3
        cute.make_layout((2, 4), stride=(1, 8)),
    )  # Apply to mode-1 layout (2,4):(1,8)

    # Apply logical divide
    result = cute.logical_divide(layout, tiler=tiler)

    # Print results
    print(">>> Layout:", layout)
    print(">>> Tiler :", tiler)
    print(">>> Logical Divide Result:", result)
    cute.printf(">?? Logical Divide Result: {}", result)


logical_divide_2d_example()

>>> Layout: (9,(4,8)):(59,(13,1))
>>> Tiler : (<cutlass.cute.core._Layout object at 0x7fa358915310>, <cutlass.cute.core._Layout object at 0x7fa358914530>)
>>> Logical Divide Result: ((3,3),((2,4),(2,2))):((177,59),((13,2),(26,1)))
>?? Logical Divide Result: ((3,3),((2,4),(2,2))):((177,59),((13,2),(26,1)))


Zipped、tiled 和 flat divide 是 `logical_divide` 的不同变体，它们可能会将模式重新排列为更方便的形式。

Zipped Divide：

In [11]:
@cute.jit
def zipped_divide_example():
    """
    演示 zipped divide：
    布局形状：(M, N, L, ...)
    Tiler 形状：<TileM, TileN>
    结果形状：((TileM,TileN), (RestM,RestN,L,...))
    """
    # Define the original layout
    layout = cute.make_layout(
        (9, (4, 8)), stride=(59, (13, 1))
    )  # (9,(4,8)):(59,(13,1))

    # Define the tiler
    tiler = (
        cute.make_layout(3, stride=3),  # Apply to mode-0 layout 3:3
        cute.make_layout((2, 4), stride=(1, 8)),
    )  # Apply to mode-1 layout (2,4):(1,8)

    # Apply zipped divide
    result = cute.zipped_divide(layout, tiler=tiler)

    # Print results
    print(">>> Layout:", layout)
    print(">>> Tiler :", tiler)
    print(">>> Zipped Divide Result:", result)
    cute.printf(">?? Zipped Divide Result: {}", result)


zipped_divide_example()

>>> Layout: (9,(4,8)):(59,(13,1))
>>> Tiler : (<cutlass.cute.core._Layout object at 0x7fa358915730>, <cutlass.cute.core._Layout object at 0x7fa358915790>)
>>> Zipped Divide Result: ((3,(2,4)),(3,(2,2))):((177,(13,2)),(59,(26,1)))
>?? Zipped Divide Result: ((3,(2,4)),(3,(2,2))):((177,(13,2)),(59,(26,1)))


Tiled Divide：

In [12]:
@cute.jit
def tiled_divide_example():
    """
    演示 tiled divide：
    布局形状：(M, N, L, ...)
    Tiler 形状：<TileM, TileN>
    结果形状：((TileM,TileN), RestM, RestN, L, ...)
    """
    # Define the original layout
    layout = cute.make_layout(
        (9, (4, 8)), stride=(59, (13, 1))
    )  # (9,(4,8)):(59,(13,1))

    # Define the tiler
    tiler = (
        cute.make_layout(3, stride=3),  # Apply to mode-0 layout 3:3
        cute.make_layout((2, 4), stride=(1, 8)),
    )  # Apply to mode-1 layout (2,4):(1,8)

    # Apply tiled divide
    result = cute.tiled_divide(layout, tiler=tiler)

    # Print results
    print(">>> Layout:", layout)
    print(">>> Tiler :", tiler)
    print(">>> Tiled Divide Result:", result)
    cute.printf(">?? Tiled Divide Result: {}", result)


tiled_divide_example()

>>> Layout: (9,(4,8)):(59,(13,1))
>>> Tiler : (<cutlass.cute.core._Layout object at 0x7fa358915310>, <cutlass.cute.core._Layout object at 0x7fa358915cd0>)
>>> Tiled Divide Result: ((3,(2,4)),3,(2,2)):((177,(13,2)),59,(26,1))
>?? Tiled Divide Result: ((3,(2,4)),3,(2,2)):((177,(13,2)),59,(26,1))


Flat Divide：

In [13]:
@cute.jit
def flat_divide_example():
    """
    演示 flat divide：
    布局形状：(M, N, L, ...)
    Tiler 形状：<TileM, TileN>
    结果形状：(TileM, TileN, RestM, RestN, L, ...)
    """
    # Define the original layout
    layout = cute.make_layout(
        (9, (4, 8)), stride=(59, (13, 1))
    )  # (9,(4,8)):(59,(13,1))

    # Define the tiler
    tiler = (
        cute.make_layout(3, stride=3),  # Apply to mode-0 layout 3:3
        cute.make_layout((2, 4), stride=(1, 8)),
    )  # Apply to mode-1 layout (2,4):(1,8)

    # Apply flat divide
    result = cute.flat_divide(layout, tiler=tiler)

    # Print results
    print(">>> Layout:", layout)
    print(">>> Tiler :", tiler)
    print(">>> Flat Divide Result:", result)
    cute.printf(">?? Flat Divide Result: {}", result)


flat_divide_example()

>>> Layout: (9,(4,8)):(59,(13,1))
>>> Tiler : (<cutlass.cute.core._Layout object at 0x7fa358915790>, <cutlass.cute.core._Layout object at 0x7fa358916030>)
>>> Flat Divide Result: (3,(2,4),3,(2,2)):(177,(13,2),59,(26,1))
>?? Flat Divide Result: (3,(2,4),3,(2,2)):(177,(13,2),59,(26,1))


### 4. Product（复制分块）

CuTe 中的 Product 操作用于根据另一个布局复制一个布局。它创建一个新布局，其中：
- 第一个模式是原始布局 A。
- 第二个模式是重新调整步长的布局 B，指向 A 的"唯一复制"的原点。

这对于在数据分块上重复线程布局以创建"重复"模式特别有用。

Logical Product（逻辑乘积）：

In [14]:
@cute.jit
def logical_product_1d_example():
    """
    演示一维逻辑乘积
    """
    # Define the original layout
    layout = cute.make_layout((2, 2), stride=(4, 1))  # (2,2):(4,1)

    # Define the tiler
    tiler = cute.make_layout(6, stride=1)  # Apply to layout 6:1

    # Apply logical product
    result = cute.logical_product(layout, tiler=tiler)

    # Print results
    print(">>> Layout:", layout)
    print(">>> Tiler :", tiler)
    print(">>> Logical Product Result:", result)
    cute.printf(">?? Logical Product Result: {}", result)


logical_product_1d_example()

>>> Layout: (2,2):(4,1)
>>> Tiler : 6:1
>>> Logical Product Result: ((2,2),(2,3)):((4,1),(2,8))
>?? Logical Product Result: ((2,2),(2,3)):((4,1),(2,8))


Blocked 和 Raked Product：
  
  - Blocked Product：以块状方式组合 A 和 B 的模式，通过在乘积后重新关联它们来保留模式的语义含义。
  - Raked Product：以交错或"耙状"方式组合 A 和 B 的模式，创建分块的循环分布。

In [15]:
@cute.jit
def blocked_raked_product_example():
    """
    演示 blocked 和 raked 乘积
    """
    # Define the original layout
    layout = cute.make_layout((2, 5), stride=(5, 1))

    # Define the tiler
    tiler = cute.make_layout((3, 4), stride=(1, 3))

    # Apply blocked product
    blocked_result = cute.blocked_product(layout, tiler=tiler)

    # Apply raked product
    raked_result = cute.raked_product(layout, tiler=tiler)

    # Print results
    print(">>> Layout:", layout)
    print(">>> Tiler :", tiler)
    print(">>> Blocked Product Result:", blocked_result)
    print(">>> Raked Product Result:", raked_result)
    cute.printf(">?? Blocked Product Result: {}", blocked_result)
    cute.printf(">?? Raked Product Result: {}", raked_result)


blocked_raked_product_example()

>>> Layout: (2,5):(5,1)
>>> Tiler : (3,4):(1,3)
>>> Blocked Product Result: ((2,3),(5,4)):((5,10),(1,30))
>>> Raked Product Result: ((3,2),(4,5)):((10,5),(30,1))
>?? Blocked Product Result: ((2,3),(5,4)):((5,10),(1,30))
>?? Raked Product Result: ((3,2),(4,5)):((10,5),(30,1))


Zipped、tiled 和 flat product：

  与 divide 操作类似，zipped、tiled 和 flat product 是 `logical_product` 的不同变体，它们可能会将模式重新排列为更方便的形式。

In [16]:
@cute.jit
def zipped_tiled_flat_product_example():
    """
    演示 zipped、tiled 和 flat 乘积
    布局形状：(M, N, L, ...)
    Tiler 形状：<TileM, TileN>

    zipped_product  : ((M,N), (TileM,TileN,L,...))
    tiled_product   : ((M,N), TileM, TileN, L, ...)
    flat_product    : (M, N, TileM, TileN, L, ...)
    """
    # Define the original layout
    layout = cute.make_layout((2, 5), stride=(5, 1))

    # Define the tiler
    tiler = cute.make_layout((3, 4), stride=(1, 3))

    # Apply zipped product
    zipped_result = cute.zipped_product(layout, tiler=tiler)

    # Apply tiled product
    tiled_result = cute.tiled_product(layout, tiler=tiler)

    # Apply flat product
    flat_result = cute.flat_product(layout, tiler=tiler)

    # Print results
    print(">>> Layout:", layout)
    print(">>> Tiler :", tiler)
    print(">>> Zipped Product Result:", zipped_result)
    print(">>> Tiled Product Result:", tiled_result)
    print(">>> Flat Product Result:", flat_result)
    cute.printf(">?? Zipped Product Result: {}", zipped_result)
    cute.printf(">?? Tiled Product Result: {}", tiled_result)
    cute.printf(">?? Flat Product Result: {}", flat_result)


zipped_tiled_flat_product_example()

>>> Layout: (2,5):(5,1)
>>> Tiler : (3,4):(1,3)
>>> Zipped Product Result: ((2,5),(3,4)):((5,1),(10,30))
>>> Tiled Product Result: ((2,5),3,4):((5,1),10,30)
>>> Flat Product Result: (2,5,3,4):(5,1,10,30)
>?? Zipped Product Result: ((2,5),(3,4)):((5,1),(10,30))
>?? Tiled Product Result: ((2,5),3,4):((5,1),10,30)
>?? Flat Product Result: (2,5,3,4):(5,1,10,30)



## 二、组合布局

### 组合布局的概念与数学表示

组合布局(Composed Layout)通过 inner/offset/outer 实现复杂数据转换：
- **inner**：布局、swizzle 或自定义变换，对坐标应用最终变换
- **offset**：整数元组，添加常量位移
- **outer**：用户可见布局，定义初始坐标变换

数学表示：$R(c) := inner(offset + outer(c))$，变换从右向左应用。

```python
layout = cute.make_composed_layout(inner, offset, outer)
```

优势：灵活性、模块化、GPU 优化访问、兼容多种变换。

### 自定义变换示例

本示例演示如何使用自定义变换函数创建组合布局。我们将创建一个简单的变换：

1. 接受 2D 坐标输入 `(x, y)`
2. 将 y 坐标加 1
3. 将其与偏移量和恒等布局组合

本示例展示如何：
- 定义自定义变换函数
- 使用变换创建组合布局
- 将布局应用于坐标
- 打印结果以进行验证

In [17]:
import cutlass
import cutlass.cute as cute
from cutlass.cute.runtime import from_dlpack, make_ptr


@cute.jit
def customized_layout():
    def inner(c):
        x, y = c
        return x, y + 1

    layout = cute.make_composed_layout(
        inner, (1, 0), cute.make_identity_layout(shape=(8, 4))
    )
    print(layout)
    cute.printf(layout(0))


customized_layout()

(<function customized_layout.<locals>.inner at 0x7fa3588e6660> o (1, 0) o (8,4):(1@0,1@1))
(1,1)


### 使用组合布局实现 Gather/Scatter 操作

Gather 使用索引数组收集元素：`output[i] = source[index[i]]`。

CuTe 实现：**offset_tensor**（索引）、**data_ptr**（源数据）、**shape**（逻辑形状）。

**工作原理**：`inner(c)` 返回 `offset_tensor[c]`；组合布局 `make_composed_layout(inner, 0, make_layout(shape))` 将坐标 `i` 映射为 `offset_tensor[i]`，实现间接访问。slice/partition 仍可应用于 outer。

**使用场景**：稀疏操作、图处理、特征嵌入、不规则数据访问。

示例打印 `i -> j`（输出索引 → 源索引）。Scatter 可类似实现（反转数据流）。

In [18]:
import torch


@cute.jit
def gather_tensor(
    offset_tensor: cute.Tensor, data_ptr: cute.Pointer, shape: cute.Shape
):
    def inner(c):
        return offset_tensor[c]

    gather_layout = cute.make_composed_layout(inner, 0, cute.make_layout(shape))
    for i in cutlass.range_constexpr(cute.size(shape)):
        cute.printf("%d -> %d", i, gather_layout(i))

    # TODO: support in future
    # gather_tensor = cute.make_tensor(data_ptr, gather_layout)
    # cute.printf(gather_tensor[0])


shape = (16,)
offset_tensor = torch.randint(0, 256, shape, dtype=torch.int32)
data_tensor = torch.arange(0, 256, dtype=torch.int32)


gather_tensor(
    from_dlpack(offset_tensor),
    make_ptr(cutlass.Int32, data_tensor.data_ptr(), cute.AddressSpace.generic),
    shape,
)

0 -> 229
1 -> 104
2 -> 222
3 -> 80
4 -> 38
5 -> 91
6 -> 157
7 -> 90
8 -> 218
9 -> 206
10 -> 91
11 -> 9
12 -> 159
13 -> 230
14 -> 171
15 -> 10
